# Dynamic Society Friction Simulator — GCP/Colab Training

**Full training pipeline with checkpoint resume support.**

This notebook:
1. Mounts Google Drive for persistent checkpoint storage
2. Clones the repo and installs dependencies
3. Generates synthetic training data (50K+ samples)
4. Fine-tunes Mistral-7B with QLoRA (LoRA r=128)
5. Auto-resumes from the last checkpoint if the session restarts
6. Syncs all checkpoints to Google Drive

**Estimated training time:** 20-50 hours on A100, 50-100 hours on T4/V100

---

## 1. Mount Google Drive & Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. Clone Repo & Install Dependencies

In [ ]:
import os

# ---- CONFIGURE THESE ----
REPO_URL = "https://github.com/YOUR_USERNAME/dynamic-society-friction-simulator.git"  # <-- CHANGE THIS
BRANCH = "main"
PROJECT_DIR = "/content/dynamic-society-friction-simulator"
# -------------------------

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print(f"Project already exists at {PROJECT_DIR}")
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}

In [ ]:
# Install all dependencies
!pip install -e ".[dev,eval]" -q

# Install flash-attn for A100 (skip if on T4/V100)
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
if "A100" in gpu_name or "H100" in gpu_name:
    print("A100/H100 detected — installing Flash Attention 2...")
    !pip install flash-attn --no-build-isolation -q
else:
    print(f"GPU: {gpu_name} — skipping Flash Attention (not supported)")

## 3. GPU-Specific Config Overrides

Auto-detects your GPU and adjusts batch size / precision if needed.

In [ ]:
import yaml
import torch

config_path = "configs/model_config.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3 if torch.cuda.is_available() else 0

print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

# Auto-adjust for smaller GPUs
if vram_gb < 20:  # T4 (16GB) or V100 (16GB)
    print("\n⚠️  Smaller GPU detected — applying T4/V100 overrides:")
    cfg["training"]["per_device_train_batch_size"] = 1
    cfg["training"]["gradient_accumulation_steps"] = 32
    cfg["training"]["max_seq_length"] = 2048
    cfg["training"]["bf16"] = False
    cfg["training"]["fp16"] = True
    cfg["training"]["optim"] = "paged_adamw_8bit"
    cfg["lora"]["r"] = 64  # reduce LoRA rank to save memory
    cfg["lora"]["lora_alpha"] = 128
    print("  - Batch size: 1 x 32 = 32 effective")
    print("  - Seq length: 2048")
    print("  - FP16 mode (no bf16)")
    print("  - LoRA r=64")
elif vram_gb < 45:  # A100 40GB
    print("\n✅ A100 40GB — using default config (optimal)")
else:  # A100 80GB
    print("\n🚀 A100 80GB — increasing batch size for faster training")
    cfg["training"]["per_device_train_batch_size"] = 8
    cfg["training"]["gradient_accumulation_steps"] = 4

# Save adjusted config
with open(config_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"\nFinal config saved to {config_path}")
print(f"  Epochs: {cfg['training']['num_train_epochs']}")
eff_batch = cfg['training']['per_device_train_batch_size'] * cfg['training']['gradient_accumulation_steps']
print(f"  Effective batch size: {eff_batch}")
print(f"  LR: {cfg['training']['learning_rate']}")
print(f"  LoRA rank: {cfg['lora']['r']}")

## 4. Generate Training Data

Generates 50K base samples → augmented to ~200K+ with perspective flips and severity scaling.

In [ ]:
import os

# Check if data already exists (from a previous run)
train_file = "data/processed/train.jsonl"
if os.path.exists(train_file):
    import json
    with open(train_file) as f:
        count = sum(1 for _ in f)
    print(f"Training data already exists: {count} samples")
    print("To regenerate, delete data/processed/ and rerun this cell.")
else:
    print("Generating training data (this may take a few minutes)...")
    !dsfs generate-data --num-samples 50000 --output-dir data/processed --seed 42

## 5. Setup Weights & Biases (Optional)

Run this cell to log training metrics to W&B. Skip if you don't want logging.

In [ ]:
# Option A: Login to W&B (recommended)
# import wandb
# wandb.login()

# Option B: Disable W&B entirely
import os
os.environ["WANDB_DISABLED"] = "true"
print("W&B disabled. Uncomment Option A above to enable.")

## 6. Train! (with auto-resume)

This cell will:
- Automatically detect if there are existing checkpoints (local or on Google Drive)
- Resume from the latest checkpoint if found
- Start fresh if no checkpoints exist
- Save checkpoints every 100 steps
- Sync checkpoints to Google Drive for persistence

**If Colab disconnects:** Just re-run cells 1-5, then this cell. It will resume automatically.

In [ ]:
from src.model.trainer import train
import logging

logging.basicConfig(level=logging.INFO)

# This will auto-resume from the latest checkpoint if one exists
trainer = train(config_path="configs/model_config.yaml")

## 7. Verify Final Model

Quick test to make sure the trained adapter works.

In [ ]:
from src.model.inference import FrictionLLM

# Load the trained model
llm = FrictionLLM(
    config_path="configs/model_config.yaml",
    adapter_path="./outputs/checkpoints/final_adapter"
)

# Test generation
test_scenario = """
A city with growing tensions between long-time residents and new immigrants.
A populist politician is blaming immigrants for rising housing costs.
Youth activists are organizing counter-protests while the media amplifies the conflict.
"""

result = llm.analyze_friction(test_scenario)
print("=" * 60)
print("MODEL OUTPUT:")
print("=" * 60)
print(result)

## 8. Copy Final Adapter to Google Drive

Save the final trained adapter to Drive so you can download it or use it later.

In [ ]:
import shutil
from pathlib import Path

src = Path("./outputs/checkpoints/final_adapter")
dst = Path("/content/drive/MyDrive/dsfs-checkpoints/final_adapter")

if src.exists():
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"Final adapter copied to Google Drive: {dst}")
    
    # Show adapter size
    total_size = sum(f.stat().st_size for f in dst.rglob('*') if f.is_file())
    print(f"Adapter size: {total_size / 1024**2:.1f} MB")
else:
    print("No final adapter found. Training may not have completed.")

## 9. Run a Quick Simulation (Optional)

Test the trained model with a short simulation run.

In [ ]:
# Run a short simulation with the trained model
!dsfs simulate \
    --adapter-path ./outputs/checkpoints/final_adapter \
    --steps 20 \
    --verbose

# Evaluate results
!dsfs evaluate --verbose